# MLP Feature Selectivity

What inputs do MLP neurons respond to? Selectivity analysis:
which neurons fire for few positions (selective) vs many (broad),
peak responses, position preferences, and output directions.

## Why This Matters

MLP feature selectivity provides tools for understanding how feed-forward layers transform and store information. Since MLPs have been shown to function as key-value memories, understanding their internal mechanics is essential for locating knowledge and understanding factual recall.

**Key references:**
- [Geva et al. (2021) "Transformer Feed-Forward Layers Are Key-Value Memories"](https://arxiv.org/abs/2012.14913) — Shows MLPs function as key-value memory stores
- [Bills et al. (2023) "Language Models Can Explain Neurons"](https://openaipublic.blob.core.windows.net/neuron-explainer/paper/index.html) — Automated neuron interpretation methods

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.mlp_feature_selectivity import (
    neuron_activation_selectivity, neuron_peak_response,
    neuron_position_preference, neuron_output_direction,
    feature_selectivity_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Activation Selectivity

Selectivity = 1 - (fraction of positions where neuron is active).
High selectivity = fires for few positions; broad = fires everywhere.

In [ ]:
for layer in range(4):
    result = neuron_activation_selectivity(model, tokens, layer=layer, top_k=5)
    print(f"  Layer {layer}: mean_selectivity={result['mean_selectivity']:.4f}")
    print(f"    Most selective:")
    for n in result['most_selective'][:3]:
        print(f"      N{n['neuron']}: sel={n['selectivity']:.3f}, active={n['active_positions']}")
    print(f"    Most broad:")
    for n in result['most_broad'][:3]:
        print(f"      N{n['neuron']}: sel={n['selectivity']:.3f}, active={n['active_positions']}")

## Peak Responses

Which neurons have the highest peak activations?

In [ ]:
for layer in range(4):
    result = neuron_peak_response(model, tokens, layer=layer, top_k=5)
    print(f"  Layer {layer}: mean_peak={result['mean_peak']:.4f}, max_peak={result['max_peak']:.4f}")
    for n in result['top_neurons'][:3]:
        print(f"    N{n['neuron']}: peak={n['peak_activation']:.4f}, "
              f"mean={n['mean_activation']:.4f}, ratio={n['peak_mean_ratio']:.2f}")

## Position Preference

Which positions does a specific neuron prefer?

In [ ]:
result = neuron_position_preference(model, tokens, layer=0, neuron=0)
print(f"Neuron L0N{result['neuron']}: peak at pos {result['peak_position']}, "
      f"{result['n_active_positions']}/{len(result['per_position'])} active")
for p in result['per_position']:
    bar = '█' * int(max(0, p['activation']) * 20)
    flag = ' *' if p['is_active'] else ''
    print(f"  Pos {p['position']}: {p['activation']:+.4f} {bar}{flag}")

## Output Directions

Each neuron's W_out column projected through W_U shows which
vocabulary items the neuron promotes or suppresses.

In [ ]:
result = neuron_output_direction(model, layer=0, top_k=5)
for n in result['per_neuron']:
    print(f"  Neuron {n['neuron']}:")
    print(f"    Promotes: {[(tok, f'{logit:.3f}') for tok, logit in n['promoted'][:3]]}")
    print(f"    Suppresses: {[(tok, f'{logit:.3f}') for tok, logit in n['suppressed'][:3]]}")

## Selectivity Summary

In [ ]:
result = feature_selectivity_summary(model, tokens)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: selectivity={p['mean_selectivity']:.4f}, "
          f"mean_peak={p['mean_peak']:.4f}")